## Ten positional embedding ma sens!!!

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$

$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

### Ten pomysł wywodzi się z tego, że:
1. numeryczny embedding tzn przypisanie koljneych liczb (0, 1, 2, ...) będzie niestabline, duże warotści będa dominować nad zwykłymi wartościami.  
2. pozycjonowanie przy pomocy wartość z [0, 1] również nie jest dobrym pomysłem, bo gdy im więcej pozycji tym częstszyy podział, nie byłoby jednolitej miary na różnych ciągach słów. 

### POMYSŁ: ZEGAR BINARY

0: 0 0 0    
1: 0 0 1   
2: 0 1 0  
3: 0 1 1  
4: 1 0 0   

pierwszy bit (WYMIAR) zmienia się często, bo co 2, każdy koljeny 2 razy rzadziej!

tutaj naturalniej do reprezenotwania tej przemiennej cykliczności nadaje się sinus i cosinus

autorzy tarnsfomera eskperymetalnie doszli do tego, że dla $i$-tego wymiaru częstotliwość:

$\omega_i = 10000^{-2i/d}$ jest odpowiednia, a każdy kolejny wymiar osycluje w okresie o również gometrycznym przyroście: $10000^{2/d}$ czyli $2\pi\cdot10000^{\frac{2\cdot0}{d}}$

wartości, są ograniczone i cykliczne, tak jak w zegarze -> wersja ciągła i różniczkowalna

korzystamy z tego tak, że $V = \text{embedding}\left(\text{token}\right) + PE(\text{pozycja tokenu})$. 

czyli $V_i = \text{embedding}\left(\text{token}\right)_i + PE(\text{pozycja tokenu}, i)$

# 1. 
Consider two positions p and q in the sequence. Write explicitly the positional encoding vectors
PE_p and PE_q for a single pair of coordinates (2i, 2i + 1).

Niech $\omega_i = 10000^{2i/d}$ wtedy

$$PE(p)_i = \begin{bmatrix} \sin(\omega_i p) \\ \cos(\omega_i p) \end{bmatrix}$$

$$PE(q)_i = \begin{bmatrix} \sin(\omega_i q) \\ \cos(\omega_i q) \end{bmatrix}$$

# 2. 
Show that the positional encoding at position p can be expressed as a linear transformation of
the positional encoding at position q that depends only on the offset ∆ = p − q.

$$PE(p)_i = \begin{bmatrix} \sin(\omega_i (q + \Delta)) \\ \cos(\omega_i (q + \Delta)) \end{bmatrix} = \begin{bmatrix} \sin(\omega_i q + \omega_i \Delta) \\ \cos(\omega_i q + \omega_i \Delta) \end{bmatrix}$$

Jak zastosujmy wzór na sin/cos sumy:
$$\sin(\omega_i q + \omega_i \Delta) = \sin(\omega_i q)\cos(\omega_i \Delta) + \cos(\omega_i q)\sin(\omega_i \Delta)$$
$$\cos(\omega_i q + \omega_i \Delta) = \cos(\omega_i q)\cos(\omega_i \Delta) - \sin(\omega_i q)\sin(\omega_i \Delta)$$

otrzymujemy postać ... macierzową tych równości ($p=q+\Delta$)

$$\begin{bmatrix} \sin(\omega_i p) \\ \cos(\omega_i p) \end{bmatrix} = \begin{bmatrix} \cos(\omega_i \Delta) & \sin(\omega_i \Delta) \\ -\sin(\omega_i \Delta) & \cos(\omega_i \Delta) \end{bmatrix} \begin{bmatrix} \sin(\omega_i q) \\ \cos(\omega_i q) \end{bmatrix}$$

# 3. 

Conclude that there exists a matrix M∆ such that
$PE_p = M_{\Delta}PE_q.$

Discuss why this property may be useful for modeling relative token positions in Transformer
architectures.

No zadanie 2 od razy daje tą macierz.

$M_\Delta$ zależy tylko od odległości $\Delta$, a nie od konkretnych wartości $p$ i $q$!

Oznacza to, że uwaga, jaką model przykłada do relacji między słowami oddalonymi o $3$ pozycje, będzie identyczna niezależnie od tego, czy te słowa znajdują się na początku, w środku, czy na końcu dokumentu.

Ta maceirz to macierz obrotu o kąt $\omega_i \Delta$

Intuicja jest właśnie taka jak pisałem wyżej, czyli każde dwie pary współrzędnych obracamy obortem: $$M_{\Delta, i} = \begin{bmatrix} \cos(\omega_i \Delta) & \sin(\omega_i \Delta) \\ -\sin(\omega_i \Delta) & \cos(\omega_i \Delta) \end{bmatrix}$$

Mamy dużo niezaleznych małych obrotów, podobnie jak w zegarze binaranym

# Teraz mamy dla słów na pozycjach p, q

$x_p = e_p + PE_p$

$x_q = e_q + PE_q$

$Q_p = x_p W_Q$, $K_p = x_p W_K$, $V_p = x_p W_V$ 

$Q_q = x_q W_Q$, $K_q = x_q W_K$, $V_q = x_q W_V$ 

Teraz podczas liczenia atencji mamy 

$$ QK^T_{(p,q)} = Q_p K^T_q = x_p W_Q W^T_K x^T_q= (e_p + PE_p) W_Q W^T_K (e_q + PE_q)^T$$


$$ QK^T_{(p,q)} = e_p W_Q W^T_K e^T_q + e_p W_Q W^T_K PE^T_q + PE_p W_Q W^T_K e^T_q + PE_p W_Q W^T_K PE^T_q$$

Zapisując $ W = W_Q W^T_K$ otrzymujemy

$$ QK^T_{(p,q)} = e_p W e^T_q + e_p W PE^T_q + PE_p W e^T_q + PE_p W PE^T_q $$

mamy tutaj 4 człowny : embedding-embedding, embedding-pozycja, pozycja-embedding, pozycja-pozycja

Wiemy, że

$$PE_p = (M_\Delta PE_q^T)^T = PE_q M_\Delta^T$$

Więc ostani człon zapisuje się jako

$$(PE_q M_\Delta^T) W PE_q^T = PE_q (M_\Delta^T W) PE_q^T$$

Sieć neuronowa podczas treningu zmienia wartości w macierzach $W_Q$ i $W_K$ tak, aby "dostroić się" do konkretnych macierzy $M_\Delta$. 

Model może na przykład nauczyć się takich wag, aby wyszedł ogromny dodatni wynik iloczynu skalarnego dokładnie wtedy, gdy $\Delta = -1$ (słowo poprzedzające), ignorując inne odległości.